<a href="https://colab.research.google.com/github/omarrgohary/Phishing-Emails-URLs-Detection/blob/main/Image.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving CEAS_08.csv to CEAS_08.csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving new_data_urls.csv to new_data_urls.csv


In [ ]:
!pip install gradio

In [ ]:
!pip install -q transformers datasets torch scikit-learn pandas torchtext tqdm

In [ ]:
import csv
import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, precision_recall_fscore_support
from transformers import DistilBertTokenizer, DistilBertModel
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
email_df = pd.read_csv(
    "/content/CEAS_08.csv",
    engine="python",
    on_bad_lines="skip"
)
print("Email shape:", email_df.shape)

Email shape: (39153, 7)


In [ ]:
LEAKAGE_PATTERNS = [
    r"\bspam\b",
    r"\bphish\b",
    r"\bphishing\b",
    r"\bham\b"
]

def remove_leakage(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    for p in LEAKAGE_PATTERNS:
        text = re.sub(p, "", text)
    return text

email_df["subject"] = email_df["subject"].apply(remove_leakage)
email_df["body"] = email_df["body"].apply(remove_leakage)
email_df["sender"] = email_df["sender"].astype(str)


In [ ]:
email_df["text_raw"] = (
    email_df["subject"].fillna("") + " " +
    email_df["body"].fillna("")
)

before = len(email_df)
email_df = email_df.drop_duplicates(subset="text_raw")
after = len(email_df)

print(f"Removed {before - after} duplicate emails")

email_df["text"] = (
    email_df["subject"].fillna("") + " [SEP] " +
    email_df["body"].fillna("")
)

email_texts = email_df["text"].tolist()
email_labels = email_df["label"].astype(int).tolist()


Removed 0 duplicate emails


In [ ]:
X_train_e, X_val_e, y_train_e, y_val_e = train_test_split(
    email_texts,
    email_labels,
    test_size=0.2,
    random_state=42,
    stratify=email_labels
)


In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_emails(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

train_enc_e = tokenize_emails(X_train_e)
val_enc_e   = tokenize_emails(X_val_e)


In [ ]:
class EmailDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


In [ ]:
class BERTEmailClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]
        cls = self.dropout(cls)
        return self.fc(cls)


In [ ]:
train_ds_e = EmailDataset(train_enc_e, y_train_e)
val_ds_e   = EmailDataset(val_enc_e, y_val_e)

train_loader_e = DataLoader(train_ds_e, batch_size=16, shuffle=True)
val_loader_e   = DataLoader(val_ds_e, batch_size=16)

email_model = BERTEmailClassifier().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(email_model.parameters(), lr=2e-5)


In [ ]:
def train_email_epoch(model, loader, epoch, total_epochs):
    model.train()
    total_loss = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [Training]", leave=False)
    for batch in pbar:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    return total_loss / len(loader)

def eval_email(model, loader, epoch, total_epochs):
    model.eval()
    preds, labels = [], []

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [Evaluating]", leave=False)
    with torch.no_grad():
        for batch in pbar:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y = batch["labels"].to(device)

            out = model(input_ids, mask)
            preds.extend(torch.argmax(out, 1).cpu().numpy())
            labels.extend(y.cpu().numpy())

    return {
        "acc": accuracy_score(labels, preds),
        "prec": precision_score(labels, preds),
        "rec": recall_score(labels, preds),
        "f1": f1_score(labels, preds)
    }


In [ ]:
EPOCHS = 2

for epoch in range(1, EPOCHS + 1):
    loss = train_email_epoch(email_model, train_loader_e, epoch, EPOCHS)
    metrics = eval_email(email_model, val_loader_e, epoch, EPOCHS)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Loss: {loss:.4f} | "
        f"Acc: {metrics['acc']:.4f} | "
        f"Prec: {metrics['prec']:.4f} | "
        f"Rec: {metrics['rec']:.4f} | "
        f"F1: {metrics['f1']:.4f}"
    )

Epoch 1/2 [Training]:   0%|          | 0/1958 [00:00<?, ?it/s]

Epoch 1/2 [Evaluating]:   0%|          | 0/490 [00:00<?, ?it/s]

Epoch 1/2 | Loss: 0.0245 | Acc: 0.9971 | Prec: 0.9973 | Rec: 0.9975 | F1: 0.9974


Epoch 2/2 [Training]:   0%|          | 0/1958 [00:00<?, ?it/s]

Epoch 2/2 [Evaluating]:   0%|          | 0/490 [00:00<?, ?it/s]

Epoch 2/2 | Loss: 0.0045 | Acc: 0.9977 | Prec: 0.9989 | Rec: 0.9970 | F1: 0.9979


In [ ]:
external_emails = [
    {
        "subject": "Security alert",
        "body": "enter the cvv code at the back of you card"
    },
    {
        "subject": "Meeting reminder",
        "body": "Just a reminder about our project meeting tomorrow at 10 AM."
    },
    {
        "subject": "University Admission Inquiry",
        "body": "Dear Admissions Office, I would like to ask about the application requirements for the MSc Computer Science program starting next fall."
    }
]

processed_texts = []
for email in external_emails:
    subj = remove_leakage(email["subject"])
    body = remove_leakage(email["body"])
    processed_texts.append(subj + " [SEP] " + body)

enc = tokenizer(
    processed_texts,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

email_model.eval()
with torch.no_grad():
    logits = email_model(
        enc["input_ids"].to(device),
        enc["attention_mask"].to(device)
    )

preds = torch.argmax(logits, dim=1).cpu().numpy()

print("\nEMAIL MODEL — REAL-WORLD TEST RESULTS\n")

for i, email in enumerate(external_emails):
    print(f"Email {i+1}")
    print("Subject:", email["subject"])
    print("Prediction:", " Phishing" if preds[i] == 1 else " Legitimate")
    print("-" * 60)



EMAIL MODEL — REAL-WORLD TEST RESULTS

Email 1
Subject: Security alert
Prediction:  Phishing
------------------------------------------------------------
Email 2
Subject: Meeting reminder
Prediction:  Legitimate
------------------------------------------------------------
Email 3
Subject: University Admission Inquiry
Prediction:  Legitimate
------------------------------------------------------------


In [ ]:
df = pd.read_csv("new_data_urls.csv")
df.head()

,url,status
0,0000111servicehelpdesk.godaddysites.com,0
1,000011accesswebform.godaddysites.com,0
2,00003.online,0
3,0009servicedeskowa.godaddysites.com,0
4,000n38p.wcomhost.com,0


In [ ]:
df = df.rename(columns={
    "url": "text",
    "status": "label"
})


In [ ]:
df['text'] = df['text'].astype(str)
df = df.dropna()


In [ ]:
df['label'] = 1 - df['label']

N_PER_CLASS = 50000

df_0 = df[df['label'] == 0].sample(N_PER_CLASS, random_state=42)
df_1 = df[df['label'] == 1].sample(N_PER_CLASS, random_state=42)

df = pd.concat([df_0, df_1]).sample(frac=1, random_state=42)
df['label'].value_counts()


,count
label,
1,50000
0,50000


In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
url_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [ ]:
def tokenize(texts):
    return url_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=64
    )

train_encodings = tokenize(train_texts.tolist())
test_encodings = tokenize(test_texts.tolist())


In [ ]:
class URLDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


In [ ]:
train_dataset = URLDataset(train_encodings, train_labels)
test_dataset = URLDataset(test_encodings, test_labels)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    do_train=True,
    do_eval=True
)


In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


/tmp/ipython-input-2147312783.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


wandb: Detected [huggingface_hub.inference, mcp] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
100,0.317500
200,0.149100
300,0.136600
400,0.139900
500,0.112800
600,0.111800
700,0.128600
800,0.113600
900,0.114800
1000,0.087700


TrainOutput(global_step=5000, training_loss=0.07648658981323242, metrics={'train_runtime': 1019.5849, 'train_samples_per_second': 156.927, 'train_steps_per_second': 4.904, 'total_flos': 2649347973120000.0, 'train_loss': 0.07648658981323242, 'epoch': 2.0})

In [ ]:
preds = trainer.predict(test_dataset)
y_pred = np.argmax(preds.predictions, axis=1)

print(classification_report(test_labels, y_pred, digits=4))

              precision    recall  f1-score   support

           0     0.9770    0.9858    0.9814     10000
           1     0.9857    0.9768    0.9812     10000

    accuracy                         0.9813     20000
   macro avg     0.9813    0.9813    0.9813     20000
weighted avg     0.9813    0.9813    0.9813     20000



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
def predict_url(url):
    model.eval()

    inputs = tokenizer(
        url,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)

    return {
        "Legitimate": probs[0][0].item(),
        "Phishing": probs[0][1].item()
    }

In [ ]:
predict_url("https://youtube.com")

{'Legitimate': 0.995134174823761, 'Phishing': 0.004865842405706644}

In [ ]:
# Save email model and tokenizer
import os
os.makedirs("/content/email_model", exist_ok=True)
torch.save(email_model.state_dict(), "/content/email_model/email_model.pt")
tokenizer.save_pretrained("/content/email_model/tokenizer")


('/content/email_model/tokenizer/tokenizer_config.json',
 '/content/email_model/tokenizer/special_tokens_map.json',
 '/content/email_model/tokenizer/vocab.txt',
 '/content/email_model/tokenizer/added_tokens.json',
 '/content/email_model/tokenizer/tokenizer.json')

In [ ]:
# Save URL model and tokenizer
model.save_pretrained("/content/url_model")
url_tokenizer.save_pretrained("/content/url_model")


('/content/url_model/tokenizer_config.json',
 '/content/url_model/special_tokens_map.json',
 '/content/url_model/vocab.txt',
 '/content/url_model/added_tokens.json',
 '/content/url_model/tokenizer.json')

In [ ]:
import torch
import torch.nn as nn
from transformers import DistilBertTokenizer, DistilBertModel, AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Email Model
email_tokenizer = DistilBertTokenizer.from_pretrained("/content/email_model/tokenizer")

class BERTEmailClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]
        cls = self.dropout(cls)
        return self.fc(cls)

email_model = BERTEmailClassifier()
email_model.load_state_dict(torch.load("/content/email_model/email_model.pt"))
email_model.to(device)
email_model.eval()

# Load URL Model
url_tokenizer = AutoTokenizer.from_pretrained("/content/url_model")
url_model = AutoModelForSequenceClassification.from_pretrained("/content/url_model")
url_model.to(device)
url_model.eval()


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [1]:
import gradio as gr
import torch
import torch.nn.functional as F

def predict(subject, body, url):
    outputs = []

    # --- Email prediction ---
    text = remove_leakage(subject) + " [SEP] " + remove_leakage(body)
    enc = tokenizer(
        [text],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    email_model.eval()
    with torch.no_grad():
        logits = email_model(enc["input_ids"], enc["attention_mask"])
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]

    label = "Phishing" if probs[1] > 0.5 else "Legitimate"
    confidence = float(probs[1] if label == "Phishing" else probs[0])
    outputs.append(f"Email Prediction: {label} ({confidence*100:.2f}%)")

    # --- URL prediction ---
    if url:
        inputs = url_tokenizer(
            [url],
            padding=True,
            truncation=True,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        url_model.eval()
        with torch.no_grad():
            logits = url_model(**inputs).logits
            probs = F.softmax(logits, dim=1).cpu().numpy()[0]

        label = "Phishing" if probs[1] > 0.5 else "Legitimate"
        confidence = float(probs[1] if label == "Phishing" else probs[0])
        outputs.append(f"URL Prediction: {label} ({confidence*100:.2f}%)")

    return "\n".join(outputs)  # return as a single string for Textbox

# --- Gradio Interface ---
interface = gr.Interface(
    fn=predict,
    inputs=[
        gr.Textbox(label="Email Subject"),
        gr.Textbox(lines=8, label="Email Body"),
        gr.Textbox(label="URL (optional)")
    ],
    outputs=[
        gr.Textbox(label="Predictions")  # Textbox instead of Label
    ],
    title="Phishing Detection System",
    description="Enter an email subject & body and/or a URL to check if it is phishing."
)

interface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ab41dfd4d40a24c755.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
